# Import Libraries & Setup

In [4]:
pip install xgboost scikit-learn pandas numpy joblib

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import json
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# 1. LOAD DATASET

print("Loading dataset...")
df = pd.read_csv(r"C:\Users\My PC\OneDrive\Documents\AI Project\used-car-price-predictor\data\pakwheels_features.csv")
print(f"Dataset loaded: {df.shape[0]:,} rows and {df.shape[1]} columns.\n")


# 2. ENCODE CATEGORICAL FEATURES

categorical_cols = [
    'make', 'model', 'assembly', 'reg_city', 'city', 
    'transmission', 'fuel_type', 'age_group', 'engine_category'
]

label_encoders = {}
for col in categorical_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

print(f"Encoded {len(categorical_cols)} categorical columns.\n")


# 3. PREPARE FEATURES AND TARGET

# Drop target-derived features to avoid data leakage
drop_cols = ['price_pkr', 'log_price', 'price_per_cc', 'price_category']
X = df.drop(drop_cols, axis=1)
y = df['price_pkr']

print(f"Features: {X.shape[1]} columns")
print(f"Target: price_pkr\n")


# 4. TRAIN/VALIDATION/TEST SPLIT (70/15/15)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42  # 0.176 * 0.85 ≈ 0.15
)

print(f"Training set:   {X_train.shape[0]:,} samples")
print(f"Validation set: {X_val.shape[0]:,} samples")
print(f"Testing set:    {X_test.shape[0]:,} samples\n")


# 5. HYPERPARAMETER TUNING

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 200, 300],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

print("Starting hyperparameter tuning with GridSearchCV...")
print(f"Testing {3*3*3*3*2*2} = 324 combinations")
print("This may take 10-15 minutes...\n")

# Base model
base_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

# Grid Search with 3-Fold Cross-Validation
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_

print("\n" + "="*70)
print("BEST HYPERPARAMETERS FOUND (XGBoost)")
print("="*70)
for param, value in best_params.items():
    print(f"  {param:20s}: {value}")
print("="*70 + "\n")


# 6. TRAIN FINAL MODEL

print("Training final XGBoost model with best parameters...")
final_model = xgb.XGBRegressor(
    **best_params,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train, y_train)
print(" Training complete!\n")


# 7. EVALUATION

def evaluate(y_true, y_pred, dataset_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"{dataset_name} Performance:")
    print(f"  MAE:  Rs {mae:,.0f}")
    print(f"  RMSE: Rs {rmse:,.0f}")
    print(f"  R²:   {r2:.4f}\n")
    
    return {'mae': float(mae), 'rmse': float(rmse), 'r2': float(r2)}

print("="*70)
print("MODEL EVALUATION")
print("="*70)

train_metrics = evaluate(y_train, final_model.predict(X_train), "Train Set")
val_metrics = evaluate(y_val, final_model.predict(X_val), "Validation Set")
test_metrics = evaluate(y_test, final_model.predict(X_test), "Test Set")

print("="*70 + "\n")


# 8. FEATURE IMPORTANCE

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features:")
print("-" * 50)
for idx, row in feature_importance.head(10).iterrows():
    print(f"  {row['feature']:25s}  {row['importance']:.6f}")
print("\n")


# 9. SAVE MODEL AND RESULTS


# Create models directory if it doesn't exist
cwd = Path.cwd()
if cwd.name == 'models':
    save_dir = cwd
else:
    save_dir = cwd / 'models'

save_dir.mkdir(parents=True, exist_ok=True)
print(f"Saving to: {save_dir}\n")

# Save model
model_path = save_dir / 'xgboost_model.pkl'
encoders_path = save_dir / 'label_encoders.pkl'
results_path = save_dir / 'xgboost_results.json'

joblib.dump(final_model, model_path)
joblib.dump(label_encoders, encoders_path)
print(f" Model saved: {model_path}")
print(f" Encoders saved: {encoders_path}")

# Save results as JSON
results = {
    'model': 'XGBoost',
    'best_params': best_params,
    'train_metrics': train_metrics,
    'val_metrics': val_metrics,
    'test_metrics': test_metrics,
    'feature_importance': feature_importance.head(15).to_dict('records')
}

with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f" Results saved: {results_path}\n")


# 10. SUMMARY

print("="*70)
print("XGBOOST TRAINING SUMMARY")
print("="*70)
print(f"Best Hyperparameters:")
for param, value in best_params.items():
    print(f"  • {param}: {value}")
print(f"\nTest Set Performance:")
print(f"  • MAE:  Rs {test_metrics['mae']:,.0f}")
print(f"  • RMSE: Rs {test_metrics['rmse']:,.0f}")
print(f"  • R²:   {test_metrics['r2']:.4f}")
print(f"\nTop 3 Features:")
print(f"  1. {feature_importance.iloc[0]['feature']}")
print(f"  2. {feature_importance.iloc[1]['feature']}")
print(f"  3. {feature_importance.iloc[2]['feature']}")
print("="*70)
print("\n XGBoost training complete!")

Loading dataset...
Dataset loaded: 24,135 rows and 26 columns.

Encoded 9 categorical columns.

Features: 22 columns
Target: price_pkr

Training set:   16,903 samples
Validation set: 3,611 samples
Testing set:    3,621 samples

Starting hyperparameter tuning with GridSearchCV...
Testing 324 = 324 combinations
This may take 10-15 minutes...

Fitting 3 folds for each of 324 candidates, totalling 972 fits

BEST HYPERPARAMETERS FOUND (XGBoost)
  colsample_bytree    : 0.8
  learning_rate       : 0.1
  max_depth           : 7
  min_child_weight    : 1
  n_estimators        : 300
  subsample           : 1.0

Training final XGBoost model with best parameters...
 Training complete!

MODEL EVALUATION
Train Set Performance:
  MAE:  Rs 256,881
  RMSE: Rs 464,041
  R²:   0.9908

Validation Set Performance:
  MAE:  Rs 515,832
  RMSE: Rs 1,449,568
  R²:   0.9072

Test Set Performance:
  MAE:  Rs 571,006
  RMSE: Rs 1,844,519
  R²:   0.8614


Top 10 Most Important Features:
----------------------------